# Quenya-TTS: development run on Kaggle

Fine-tunes the pretrained multilingual ToucanTTS model on Quenya recordings, on Kaggle's free GPU. This is for development and experiments on recordings you may use yourself: **volunteer recordings never go to Kaggle.** The public model is trained on Gcore with `python -m quenya_tts.gcore train` (see the README).

Before running:
1. **Settings → Accelerator → GPU T4 x2** (one GPU is used), and **Settings → Internet → on** (needs a phone-verified Kaggle account). Keep the notebook **private**, and turn on 2-step verification for your Kaggle account.
2. **Add Input → your dataset**, laid out as `mp3/<name>.mp3` and `texts/<name>.txt` (one line of Quenya per line, in the order it is read). Recordings whose licence does not allow publishing a model trained on them (such as Glǽmscrafu, for now) give a model for experiments only.

**Where the data goes.** The clips, Toucan's features and the checkpoints stay in `/kaggle/temp`, which Kaggle never saves and deletes when the session ends. Only logs, scores and watermarked samples are written to `/kaggle/working` (the saved output).


In [ ]:
import glob, os
STEPS = 500                                     # 500 for a first test run; a few thousand for a real one
KEEP_CHECKPOINT = False                         # True copies best.pt to the output
TEMP = "/kaggle/temp"                           # never saved, deleted with the session
OUT = "/kaggle/working"                         # saved output
REPO = f"{TEMP}/Quenya-TTS"
PY = f"{TEMP}/py311/bin/python"                 # Toucan needs Python 3.10/3.11; Kaggle's own Python is newer
os.makedirs(TEMP, exist_ok=True)
# The notebook's inline plot backend does not exist in the training environment; Toucan only saves plots to files.
os.environ["MPLBACKEND"] = "Agg"
# setuptools' distutils shim trips over the uv Python when triton imports it in Toucan's worker processes.
os.environ["SETUPTOOLS_USE_DISTUTILS"] = "stdlib"
# The clips cut from the recordings go to the temporary disk too.
os.environ["QUENYA_CORPORA_DIR"] = f"{TEMP}/corpora"

# Kaggle mounts datasets at different depths (/kaggle/input/<name> or /kaggle/input/datasets/<owner>/<name>),
# and an upload may keep or flatten its folders, so look for the recordings and the texts anywhere under it.
def folder_with(extension):
    files = glob.glob(f"/kaggle/input/**/*{extension}", recursive=True)
    if not files:
        raise FileNotFoundError(f"no {extension} file under /kaggle/input: add the dataset with Add Input")
    return os.path.dirname(sorted(files)[0])
AUDIO, TEXTS = folder_with(".mp3"), folder_with(".txt")
print("recordings:", AUDIO, len(glob.glob(f"{AUDIO}/*.mp3")), "| texts:", TEXTS, len(glob.glob(f"{TEXTS}/*.txt")))
CORPUS = "Glaemscrafu"                          # corpus name, as quenya_tts.train expects it


## 1. Install

In [ ]:
%%bash
set -e
TEMP=/kaggle/temp  # as TEMP in the first cell
apt-get -qq update > /dev/null 2>&1 || true
apt-get -qq install -y espeak-ng libportaudio2 > /dev/null
cd "$TEMP"  # a rerun may start inside the folder deleted below
rm -rf "$TEMP/Quenya-TTS" "$TEMP/py311"  # start clean when the cell is run again
git clone -q --recurse-submodules https://github.com/brunopacheco1/Quenya-TTS.git "$TEMP/Quenya-TTS"
cd "$TEMP/Quenya-TTS"
# Toucan pins packages (torch 2.4, numpy 1.23, librosa 0.9) that have no builds for Kaggle's Python, so everything
# goes into a separate Python 3.11 environment, made with uv. The GUI-only packages are left out: training does
# not need them and some do not install on a headless machine. audioseal watermarks generated audio.
pip install -q uv
uv venv -q --seed --python 3.11 "$TEMP/py311"  # --seed adds pip, which phonepiece imports
grep -viE '^(pyqt5|pyqtgraph|pynput|PyAutoGUI|gradio)' requirements.txt > /tmp/requirements.txt
uv pip install -q --python "$TEMP/py311/bin/python" -r /tmp/requirements.txt "transformers>=4.46,<4.50" audioseal
"$TEMP/py311/bin/python" -c "import torch; print('torch', torch.__version__, 'GPU:', torch.cuda.is_available())"

## 2. Cut the recordings into clips

The recordings are cut into one clip per line; lines the phone recogniser hears too differently from the text are left out, and `cuts.csv` lists every line with its phone error rate.


In [ ]:
%cd {REPO}
!{PY} -m quenya_tts.prepare_corpus --audio "{AUDIO}" --texts "{TEXTS}" --name {CORPUS} --device cuda
import pandas as pd
cuts = pd.read_csv(f"{os.environ['QUENYA_CORPORA_DIR']}/{CORPUS}/cuts.csv")
print(cuts.kept.value_counts())
display(cuts.sort_values("per", ascending=False).head(15))


## 3. Train

In [ ]:
!{PY} -m quenya_tts.train --gpu_id 0 --corpora {CORPUS} --steps {STEPS} --batch_size 8 2>&1 | grep -v Warning | tee {OUT}/train.log

## 4. The trained model

A model trained here is never released: releases are made from the Gcore training of the volunteer recordings.


In [ ]:
best = sorted(glob.glob(f"{REPO}/Models/ToucanTTS_Quenya*/best.pt"))[-1]
MODEL = best
if KEEP_CHECKPOINT:
    !cp {best} {OUT}/best.pt


## 5. Listen and score

Reads four test sentences with the pretrained model (before) and the new one (after), then lets the phone recogniser score both: the phone error rate (PER) is how far what it hears is from the phones our rules expect, so lower is better. Every reading carries the AI-generated watermark; the last line checks it.

In [ ]:
from IPython.display import Audio, display
tests = {
    "elen_sila": "Elen síla lúmenn' omentielvo.",
    "ilye": "Ar ilyë tier undulávë lumbulë.",
    "valimar": "Namárië! Nai hiruvalyë Valimar.",
    "dialogue": "Ma quetilyë Quenya? Quetin Quenya.",
}
for name, text in tests.items():
    !{PY} -m quenya_tts.read "{text}" --device cuda -o {OUT}/{name}_before.wav 2>/dev/null
    !{PY} -m quenya_tts.read "{text}" --model {MODEL} --device cuda -o {OUT}/{name}.wav 2>/dev/null
    print(text)
    display(Audio(f"{OUT}/{name}_before.wav"), Audio(f"{OUT}/{name}.wav"))

score = """
import sys, librosa
from quenya_g2p import to_ipa
from quenya_tts.compare import SR, ipa_to_phones, load_recognizer, phone_error_rate
out = sys.argv[1]
recognize = load_recognizer("cuda")
for name, text in zip(sys.argv[2::2], sys.argv[3::2]):
    expected = ipa_to_phones(to_ipa(text))
    for version in ("before", "after"):
        path = f"{out}/{name}_before.wav" if version == "before" else f"{out}/{name}.wav"
        heard = recognize(librosa.load(path, sr=SR)[0])
        per, _ = phone_error_rate(expected, ipa_to_phones(heard))
        print(f"{name:10} {version:6} PER {per:.2f}  heard: {heard}")
"""
open("/tmp/score.py", "w").write(score)
args = " ".join(f'{name} "{text}"' for name, text in tests.items())
!PYTHONPATH=. {PY} /tmp/score.py {OUT} {args} 2>/dev/null | tee {OUT}/scores.txt
!{PY} -m quenya_tts.watermark check {OUT}/elen_sila.wav 2>/dev/null

## 6. Keep the results

Download from the Output tab: `train.log`, `scores.txt` and the samples. Toucan's progress plots stay with the checkpoints in the temporary folder; copy them out with `!cp {REPO}/Models/ToucanTTS_Quenya*/*.png {OUT}` if you want them.